In [ ]:
"""
Example script to run the electricity model.
It consists of three parts:
1. Generation
2. Grid
3. Storage
"""

import pandas as pd
import numpy as np
import xarray as xr
import time
import warnings
from pathlib import Path
import warnings
from pint.errors import UnitStrippedWarning
warnings.simplefilter("ignore", UnitStrippedWarning)

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA

from imagematerials.model import GenericStocks, MaterialIntensities, SharesInflowStocks
from imagematerials.factory import ModelFactory, Sector
from imagematerials.preprocessing import get_preprocessing_data
import prism


# TODO absolute path of file "preprocessing.py" ? current solution can differ depending on IDE used (?) 
path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_base = Path(path_base, "data", "raw")

YEAR_START = 1971   # start year of the simulation period
YEAR_END = 2100     # end year of the calculations
YEAR_OUT = 2100     # year of output generation = last year of reporting

# Combined -------------------------------------------------------------

In [ ]:
start_time = time.time()
# generation
prep_data = get_preprocessing_data_gen(path_base, SCEN, VARIANT, YEAR_START, YEAR_END, YEAR_OUT)
sec_electr_gen = Sector("electr_gen", prep_data)
# grid
prep_data_lines, prep_data_add = get_preprocessing_data_grid(path_base, SCEN, VARIANT, YEAR_START, YEAR_END, YEAR_OUT)
sec_electr_grid_lines = Sector("electr_grid_lines", prep_data_lines)
sec_electr_grid_add = Sector("electr_grid_add", prep_data_add)
# storage
prep_data_phs, prep_data_oth_storage = get_preprocessing_data_stor(path_base, SCEN, VARIANT, YEAR_START, YEAR_END, YEAR_OUT)
sec_electr_stor_phs = Sector("electr_stor_phs", prep_data_phs)
sec_electr_stor_oth = Sector("electr_stor_oth", prep_data_oth_storage, check_coordinates=False)

time_start = prep_data["stocks"].coords["Time"].min().values
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)


factory = ModelFactory(
    [sec_electr_gen, 
    sec_electr_grid_lines, 
    sec_electr_grid_add,
    sec_electr_stor_phs,
    sec_electr_stor_oth], complete_timeline
    ).add(GenericStocks, ["electr_gen", 
                            "electr_grid_lines",
                            "electr_grid_add",
                            "electr_stor_phs"
                            ] 
    ).add(SharesInflowStocks, "electr_stor_oth"
    ).add(MaterialIntensities, ["electr_gen", 
                            "electr_grid_lines",
                            "electr_grid_add",
                            "electr_stor_phs",
                            "electr_stor_oth"
                            ]
    ).finish()

factory.simulate(simulation_timeline)

list(factory.electr_gen)

end_time = time.time()
print(f"Execution time: {end_time - start_time:.4f} seconds")

In [ ]:
factory.stocks["electr_gen"]#.Regions

In [ ]:
factory.stocks.keys()
# factory.stocks["electr_stor_oth"]


# Combined run - all electricity subsectors

In [ ]:
scenario_list = {"SSP2_M_CP":("SSP2_M_CP", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA


    elc_sector = get_preprocessing_data("electricity", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    
    factory = ModelFactory(
    elc_sector, complete_timeline
    ).add(GenericStocks, ["elc_gen",
                         "elc_grid_lines",
                         "elc_grid_add",
                         "elc_stor_phs"
                          ]
    ).add(SharesInflowStocks, ["elc_stor_other"],
    ).add(MaterialIntensities, ["elc_gen",
                         "elc_grid_lines",
                         "elc_grid_add",
                         "elc_stor_phs",
                         "elc_stor_other"
                          ]
    )
    model = factory.finish()

    model.simulate(simulation_timeline)
    print(f"Finished {scen_id}")

# Generation -------------------------------------------------------

In [ ]:
scenario_list = {#"SSP2_M_CP":("SSP2_M_CP", None),
                #  "SSP2_narrow":("SSP2_narrow", ["narrow"]),
                "SSP2_slow":("SSP2_slow", ["slow"]),
                 }

time_start = 1971
complete_timeline = prism.Timeline(time_start, 2030, 1) #YEAR_END
simulation_timeline = prism.Timeline(YEAR_START, 2030, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    
    # Paths to scenario data
    print(climate_scen)
    print(circular_scen)
    if circular_scen is not None:
        climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
        circular_economy_scenario_dirs = {
            scenario: Path(path_base, "circular_economy_scenarios", scenario) for scenario in circular_scen # path to circular economy scenario data
        }
    else:
        climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
        circular_economy_scenario_dirs = None
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    elc_sector = get_preprocessing_data("electricity", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs, cache = False) 
    # elc_sector is a list of preprocessing data for each electricity subsector

    elc_sector_gen = next(item for item in elc_sector if item.name == "elc_gen")

    factory = ModelFactory(
        elc_sector_gen, complete_timeline
        ).add(GenericStocks
        ).add(MaterialIntensities
        )
    model = factory.finish()

    model.simulate(simulation_timeline)
    print(f"Finished {scen_id}")
    list(model.elc_gen)

# Grid -------------------------------------------------------------

## Lines ----

In [ ]:
scenario_list = {#"SSP2_M_CP":("SSP2_M_CP", None),
                "SSP2_narrow":("SSP2_narrow", ["narrow"]),
                 }

time_start = 1971
complete_timeline = prism.Timeline(time_start, 2030, 1) #YEAR_END
simulation_timeline = prism.Timeline(YEAR_START, 2030, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    
    # Paths to scenario data
    print(climate_scen)
    print(circular_scen)
    if circular_scen is not None:
        climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
        circular_economy_scenario_dirs = {
            scenario: Path(path_base, "circular_economy_scenarios", scenario) for scenario in circular_scen # path to circular economy scenario data
        }
    else:
        climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
        circular_economy_scenario_dirs = None
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    elc_sector = get_preprocessing_data("electricity", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs, cache = False) 
    # elc_sector is a list of preprocessing data for each electricity subsector

    elc_sector_grid_lines = next(item for item in elc_sector if item.name == "elc_grid_lines")

    factory = ModelFactory(
        elc_sector_grid_lines, complete_timeline
        ).add(GenericStocks
        ).add(MaterialIntensities
        )
    model_lines = factory.finish()

    model_lines.simulate(simulation_timeline)
    print(f"Finished {scen_id}")
    list(model_lines.elc_grid_lines)

## Grid Additions ----

In [ ]:
scenario_list = {#"SSP2_M_CP":("SSP2_M_CP", None),
                #"SSP2_narrow":("SSP2_narrow", ["narrow"]),
                "SSP2_slow":("SSP2_slow", ["slow","narrow"]),
                 }

time_start = 1971
complete_timeline = prism.Timeline(time_start, 2030, 1) #YEAR_END
simulation_timeline = prism.Timeline(YEAR_START, 2030, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    
    # Paths to scenario data
    print(climate_scen)
    print(circular_scen)
    if circular_scen is not None:
        climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
        circular_economy_scenario_dirs = {
            scenario: Path(path_base, "circular_economy_scenarios", scenario) for scenario in circular_scen # path to circular economy scenario data
        }
    else:
        climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
        circular_economy_scenario_dirs = None
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    elc_sector = get_preprocessing_data("electricity", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs, cache = False) 
    # elc_sector is a list of preprocessing data for each electricity subsector

    elc_sector_grid_add = next(item for item in elc_sector if item.name == "elc_grid_add")

    factory = ModelFactory(
        elc_sector_grid_add, complete_timeline
        ).add(GenericStocks
        ).add(MaterialIntensities
        )
    model_add = factory.finish()

    model_add.simulate(simulation_timeline)
    print(f"Finished {scen_id}")
    list(model_add.elc_grid_add)

# Storage -------------------------------------------------------

In [ ]:
climate_scen = "SSP2_M_CP"
circular_scen = None

time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
# climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA


elc_sector = get_preprocessing_data("electricity", path_base, climate_policy_scenario_dir, circular_economy_scenario_dir, cache = False) 
# elc_sector is a list of preprocessing data for each electricity subsector

In [ ]:
# Pumped Hydropower Storage (PHS) =====

elc_sector_stor_phs = next(item for item in elc_sector if item.name == "elc_stor_phs")

factory = ModelFactory(
    elc_sector_stor_phs, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model_phs = factory.finish()

model_phs.simulate(simulation_timeline)
list(model_phs.elc_stor_phs)

In [ ]:
# Other Storage =====

elc_sector_stor_other = next(item for item in elc_sector if item.name == "elc_stor_other")
# sec_electr_stor_oth = Sector("electr_stor_oth", prep_data_oth_storage, check_coordinates=False)

factory = ModelFactory(
    elc_sector_stor_other, complete_timeline
    ).add(SharesInflowStocks
    ).add(MaterialIntensities
    )
model_other = factory.finish()

model_other.simulate(simulation_timeline)
list(model_other.elc_stor_other)

## Visualizations

In [ ]:
import matplotlib.pyplot as plt

test_da = prep_data_oth_storage["shares"]

# Example dictionary mapping technologies to subcategories
tech_map_Sebastiaan = {
    'NiMH': 'Other',
    'LFP': 'Lithium-ion',
    'LMO': 'Lithium-ion',
    'LTO': 'Lithium-ion',
    'NMC': 'Lithium-ion',
    'NCA': 'Lithium-ion',
    'Lithium Sulfur': 'Advanced Li',
    'Lithium Ceramic': 'Advanced Li',
    'Lithium-air': 'Advanced Li',
    'Flywheel': 'Flywheel',
    'Compressed Air': 'Compressed Air',
    'Hydrogen FC': 'Hydrogen',
    'Zinc-Bromide': 'Flow Batteries',
    'Vanadium Redox': 'Flow Batteries',
    'Sodium-Sulfur': 'Molten-salt Batteries',
    'ZEBRA': 'Molten-salt Batteries',
    'Deep-cycle Lead-Acid': "Other",
}

# Color dictionary for categories
color_map = {
    'Lithium-ion':      '#F0BA3C',
    'Advanced Li':      '#EDCD76',
    'Flywheel':         '#4387F6',
    'Compressed Air':   '#BABAB8',
    'Hydrogen':         '#6A26D1',
    'Flow Batteries':   '#5C55D6',
    'Molten-salt Batteries': '#31942A',
    'Other':            '#A72C41',
}
category_order = ['Flywheel', 'Compressed Air', 'Lithium-ion', 'Advanced Li', 'Flow Batteries', 'Molten-salt Batteries', 'Hydrogen', 'Other']
category_order = ['Other', 'Hydrogen','Molten-salt Batteries', 'Flow Batteries', 'Advanced Li', 'Lithium-ion', 'Compressed Air','Flywheel' ]

# Get the types from your DataArray
types = test_da.Type.values  # da is your xarray.DataArray
# Map each type to its category
categories = [tech_map_Sebastiaan[t] for t in types]
# Assign as a new coordinate or DataArray
da_grouped = test_da.assign_coords(Category=('Type', categories))

# Sum over technologies within the same category
category_sum = da_grouped.groupby('Category').sum(dim='Type')
# Convert to pandas for plotting
df = category_sum.to_pandas()
df = df[category_order]

fig, ax = plt.subplots(figsize=(13, 8))
# Plot stacked bar with defined colors
df.loc[1990:2060,:].plot.area(ax=ax, stacked=True, color=[color_map[c] for c in df.columns])
plt.xlabel('Year')  # or 'Cohort'
plt.ylabel('Share')
plt.title('Technology Categories over Time')
plt.legend(title='Category')
plt.tight_layout()
# fig.savefig(path_test_plots / f"Market_shares.png", dpi=300, bbox_inches='tight')
plt.show()
